# Analysis Experiments

Test prompts/models for generating call analysis. This notebook demonstrates the analysis helpers - feel free to use different models (or combine them) for different parts of the analysis.

**Key points:**
- Topics & keywords auto-created if they don't exist (normalized to lowercase)
- Output fields: `summary`, `sentiment_score`, `sentiment_label`, `key_moves`, `is_resolved`, `topics`, `keywords`

In [ ]:
%pip install openai

In [ ]:
import json
import sys
from pathlib import Path

backend = Path.cwd().parent.parent
sys.path.insert(0, str(backend))

from openai import OpenAI

OPENAI_API_KEY = "" # <-- Paste your key here
openai_client = OpenAI(api_key=OPENAI_API_KEY)

## 1. Load transcript

In [ ]:
# # Option A: From positive.json file
# with open(backend / "transcripts" / "positive.json") as f:
#     samples = json.load(f)

# transcript = samples[0]["transcript"]["turns"]
# print(f"Loaded {len(transcript)} turns")
# print(transcript)

In [ ]:
## Option B: From SUPABASE
import os
from dotenv import load_dotenv
from supabase import create_client
from database.constants import Tables

load_dotenv(backend.parent / ".env")
supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_ROLE_KEY"])
result = supabase.table(Tables.CALLS).select("id, transcript").not_.is_("transcript", "null").limit(1).execute()
call_id, transcript = result.data[0]["id"], result.data[0]["transcript"]
print(f"Loaded call {call_id} with {len(transcript)} turns")
print(transcript)

## 2. Prompt & model config

In [ ]:
TOPICS = ["billing", "refund", "subscription", "cancellation", "technical support",
          "account setup", "password reset", "shipping", "returns", "complaint"]

KEYWORDS = ["frustrated", "angry", "satisfied", "confused", "urgent", "manager", 
            "escalate", "cancel", "refund", "charge", "wait", "delay", "resolved", "issue"]

SYSTEM_PROMPT = f"""
Analyze this call transcript. Return JSON with:
- summary: 2-3 sentence summary
- sentiment_score: -1.0 (negative) to 1.0 (positive)
- sentiment_label: "positive", "neutral", or "negative"
- key_moves: list of agent techniques used
- is_resolved: boolean
- topics: pick from {TOPICS} (or add new if needed)
- keywords: pick from {KEYWORDS} (or add new if needed)
""".strip()

MODEL = "gpt-4o-mini"

In [ ]:
def format_transcript(turns):
    return "\n".join(f"{t['speaker']}: {t['text']}" for t in turns)

def analyze_transcript(turns, model=MODEL):
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, 
                  {"role": "user", "content": format_transcript(turns)}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

## 3. Run analysis

In [ ]:
result = analyze_transcript(transcript)
print(json.dumps(result, indent=2))

## 4. Compare models (optional)

In [ ]:
# models = ["gpt-4o-mini", "gpt-4o"]
# for model in models:
#     res = analyze_transcript(transcript, model=model)
#     print(f"\n=== {model} ===")
#     print(f"Sentiment: {res['sentiment_score']} ({res['sentiment_label']})")
#     print(f"Topics: {res['topics']}")

## 5. Save to database (optional)

**Note:** This DELETES ANY EXISTING ANALYSIS for the call before saving (only one analysis per call allowed).
**MAKE SURE YOU USED THE SUPABASE OPTION IN PART 1 BEFORE UNCOMMENTING THIS SO `CALL_ID` GETS DEFINED**

In [ ]:
import os
from dotenv import load_dotenv
from supabase import create_client
from database import analysis
from database.constants import Tables

load_dotenv(backend.parent / ".env")
supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_ROLE_KEY"])

# Delete existing analysis for this call (if any)
supabase.table(Tables.CALL_ANALYSES).delete().eq("call_id", call_id).execute()

saved = analysis.create_analysis(supabase, call_id=call_id,
    summary=result["summary"], sentiment_score=result["sentiment_score"],
    sentiment_label=result["sentiment_label"], key_moves=result["key_moves"],
    is_resolved=result["is_resolved"])

analysis.add_topics_to_analysis(supabase, saved["id"], result["topics"])
analysis.add_keywords_to_analysis(supabase, saved["id"], result["keywords"])

print(f"Saved analysis - id: {saved['id']}")
print(json.dumps(analysis.get_analysis_by_call_id(supabase, call_id), indent=2))